In [1]:
import pandas as pd
import os
from IPython.display import display
import numpy as np

# --- CONFIGURAÇÕES ---
base_project_path = os.getcwd() 
print(f"Diretório base do script: {base_project_path}")

# Definição dos caminhos conforme sua estrutura
# Script está em: .../normal
# Argus CSVs em: .../normal/logs
# Zeek Logs em: .../normal/logs/logs

argus_dir = os.path.join(base_project_path, 'logs')
zeek_dir = os.path.join(base_project_path, 'logs', 'logs')

# Defina os RÓTULOS para esta categoria (Normal = 0)
label_value = 1

print(f"--- Processando Categoria: Normal ---")
print(f"Diretório Zeek: {zeek_dir}")
print(f"Diretório Argus: {argus_dir}")

# --- FUNÇÕES ---

def read_zeek_log(file_path):
    """Lê um arquivo de log Zeek e retorna um DataFrame."""
    columns = []
    try:
        with open(file_path, 'r', encoding='latin-1') as f:
            for line in f:
                if line.startswith('#fields'):
                    columns = line.strip().split('\t')[1:]
                    break
        if not columns:
            print(f"Aviso: Não foi encontrada a linha #fields em {file_path}")
            return pd.DataFrame()

        df = pd.read_csv(
            file_path,
            sep='\t',
            names=columns,
            comment='#',
            header=None,
            low_memory=False,
            na_values=['-', '(empty)'],
            encoding='latin-1'
        )
        return df
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")
        return pd.DataFrame()

def load_and_prepare_argus(argus_path):
    """Carrega CSV do Argus e padroniza para merge com Zeek."""
    if not os.path.exists(argus_path):
        print(f"AVISO: Arquivo Argus não encontrado em {argus_path}")
        return None
    
    print(f"Carregando dados do Argus: {argus_path}")
    df_argus = pd.read_csv(argus_path, low_memory=False)
    
    # 1. Limpeza dos nomes das colunas
    df_argus.columns = df_argus.columns.str.strip()

    # 2. Mapeamento Argus -> Zeek/UNSW
    rename_map = {
        'SrcAddr': 'id.orig_h',
        'DstAddr': 'id.resp_h',
        'Sport':   'id.orig_p',
        'Dport':   'id.resp_p',
        'Proto':   'proto',
        'State':   'state',
        'Dur':     'dur',
        'sTtl':    'sttl',
        'dTtl':    'dttl',
        'SrcLoss': 'sloss',
        'DstLoss': 'dloss',
        'SrcPkts': 'spkts',
        'DstPkts': 'dpkts',
        'SrcWin':  'swin',
        'DstWin':  'dwin',
        'SrcTCPBase': 'stcpb',
        'DstTCPBase': 'dtcpb',
        'sMeanPktSz': 'smeansz',
        'dMeanPktSz': 'dmeansz',
        'SrcJitter':  'sjit',
        'DstJitter':  'djit',
        'SIntPkt':    'sinpkt',
        'DIntPkt':    'dinpkt',
        'TcpRtt':     'tcprtt',
        'SynAck':     'synack',
        'AckDat':     'ackdat',
        'SrcBytes':   'sbytes',
        'DstBytes':   'dbytes'
    }
    
    # Renomeia apenas o que existe
    actual_rename = {k: v for k, v in rename_map.items() if k in df_argus.columns}
    df_argus.rename(columns=actual_rename, inplace=True)
    
    # 3. Verificação de Segurança (Chaves de Merge)
    required_keys = ['id.orig_h', 'id.resp_h', 'id.orig_p', 'id.resp_p', 'proto']
    missing = [k for k in required_keys if k not in df_argus.columns]
    if missing:
        print(f"ERRO CRÍTICO: Colunas chave faltando no Argus após rename: {missing}")
        return None

    # Nota: A conversão estrita de tipos agora é feita no bloco principal de merge para garantir simetria com Zeek
    
    return df_argus

# --- EXECUÇÃO PRINCIPAL ---
log_data = {} 
log_types_info = {
    'conn': {'required': True, 'columns': None},
    'http': {'required': False, 'columns': ['uid', 'trans_depth', 'response_body_len', 'method']},
    'ftp': {'required': False, 'columns': ['uid', 'user', 'password', 'command']},
    'dns': {'required': False, 'columns': ['uid', 'query']}
}

if not os.path.isdir(zeek_dir):
    print(f"ERRO CRÍTICO: Diretório de logs Zeek não encontrado - {zeek_dir}")
else:
    all_files_in_dir = os.listdir(zeek_dir)

    # 1. Leitura dos Logs Zeek (Na pasta ./logs/logs)
    for log_type, info in log_types_info.items():
        relevant_files = [f for f in all_files_in_dir if f.startswith(log_type + '.') and f.endswith('.log')]

        if not relevant_files:
            if info['required']:
                print(f"ERRO CRÍTICO: {log_type}.log necessário não encontrado em {zeek_dir}.")
            log_data[log_type] = pd.DataFrame()
            continue

        df_list = [read_zeek_log(os.path.join(zeek_dir, f)) for f in relevant_files]
        combined_df = pd.concat(df_list, ignore_index=True)

        if info['columns']:
            cols_to_keep = [col for col in info['columns'] if col in combined_df.columns]
            if 'uid' not in cols_to_keep and 'uid' in combined_df.columns:
                 cols_to_keep.insert(0, 'uid')
            if cols_to_keep:
                 combined_df = combined_df[cols_to_keep]

        log_data[log_type] = combined_df
        print(f"Lidos {len(relevant_files)} logs de {log_type}. Total linhas: {len(combined_df)}")


    # 2. Junção dos Logs Zeek (Merge por UID)
    if 'conn' in log_data and not log_data['conn'].empty:
        final_df = log_data['conn'].copy()
        
        for log_type in ['http', 'ftp', 'dns']:
            if log_type in log_data and not log_data[log_type].empty and 'uid' in log_data[log_type].columns:
                log_data[log_type] = log_data[log_type].drop_duplicates(subset=['uid'], keep='first')
                cols_to_rename = {col: f"{log_type}_{col}" for col in log_data[log_type].columns if col != 'uid'}
                df_to_merge = log_data[log_type].rename(columns=cols_to_rename)
                final_df = pd.merge(final_df, df_to_merge, on='uid', how='left')

        # 3. Junção com Argus (Múltiplos CSVs)
        
        # Lista de arquivos para carregar (Adicionar os CSVs aqui)
        argus_filename_list = [
            "captura_exploit1.csv",
            "captura_exploit2.csv"
        ]
        
        argus_dfs_list = []
        
        print("\nIniciando carregamento dos arquivos Argus...")
        for fname in argus_filename_list:
            full_argus_path = os.path.join(argus_dir, fname)
            df_temp = load_and_prepare_argus(full_argus_path)
            if df_temp is not None:
                argus_dfs_list.append(df_temp)
        
        if argus_dfs_list:
            # Concatena todos os DataFrames do Argus em um único Grande DataFrame
            df_argus_data = pd.concat(argus_dfs_list, ignore_index=True)
            
            # --- CORREÇÃO DE MERGE ROBUSTO (PADRONIZAÇÃO DE CHAVES) ---
            print("\nPADRONIZANDO CHAVES DE MERGE ZEEK <-> ARGUS...")
            
            # Função auxiliar para garantir inteiros em portas (trata '80.0' e '80')
            def clean_port(val):
                try:
                    return int(float(val))
                except:
                    return 0

            # 1. Limpeza no Zeek (final_df)
            final_df['id.orig_p'] = final_df['id.orig_p'].apply(clean_port)
            final_df['id.resp_p'] = final_df['id.resp_p'].apply(clean_port)
            final_df['proto'] = final_df['proto'].astype(str).str.lower().str.strip()
            final_df['id.orig_h'] = final_df['id.orig_h'].astype(str).str.strip()
            final_df['id.resp_h'] = final_df['id.resp_h'].astype(str).str.strip()

            # 2. Limpeza no Argus (df_argus_data)
            df_argus_data['id.orig_p'] = df_argus_data['id.orig_p'].apply(clean_port)
            df_argus_data['id.resp_p'] = df_argus_data['id.resp_p'].apply(clean_port)
            df_argus_data['proto'] = df_argus_data['proto'].astype(str).str.lower().str.strip()
            df_argus_data['id.orig_h'] = df_argus_data['id.orig_h'].astype(str).str.strip()
            df_argus_data['id.resp_h'] = df_argus_data['id.resp_h'].astype(str).str.strip()

            merge_keys = ['id.orig_h', 'id.resp_h', 'id.orig_p', 'id.resp_p', 'proto']

            # Diagnóstico de Interseção (Para verificar se o merge vai funcionar)
            # Cria chaves compostas temporárias apenas para contar match
            zeek_keys = set(zip(final_df['id.orig_h'], final_df['id.resp_h'], final_df['id.orig_p'], final_df['id.resp_p'], final_df['proto']))
            argus_keys = set(zip(df_argus_data['id.orig_h'], df_argus_data['id.resp_h'], df_argus_data['id.orig_p'], df_argus_data['id.resp_p'], df_argus_data['proto']))
            match_count = len(zeek_keys.intersection(argus_keys))
            print(f"DIAGNÓSTICO: Encontrados {match_count} fluxos coincidentes exatos entre Zeek e Argus.")

            # Remove duplicatas no Argus antes do merge
            df_argus_data.drop_duplicates(subset=merge_keys, keep='first', inplace=True)
            
            print(f"Total de registros Argus consolidados para merge: {len(df_argus_data)}")
            print("\nExecutando merge Zeek + Argus...")
            
            # Identifica colunas extras do Argus
            cols_to_use = [c for c in df_argus_data.columns if c not in merge_keys]
            
            # Merge Left (Prioridade Zeek, enriquecido com Argus)
            final_df = pd.merge(
                final_df,
                df_argus_data[merge_keys + cols_to_use],
                on=merge_keys,
                how='left',
                suffixes=('', '_argus')
            )
            print(f"Merge Argus concluído.")

            # --- BLOCO DE PRIORIDADE E CONSOLIDAÇÃO ---
            # Sobrescreve colunas base do Zeek com Argus (que tem sttl, etc.)
            consolidation_map = {
                'dur': 'duration',      # Argus 'dur' -> Zeek 'duration'
                'sbytes': 'orig_bytes', # Argus 'sbytes' -> Zeek 'orig_bytes'
                'dbytes': 'resp_bytes',
                'spkts': 'orig_pkts',
                'dpkts': 'resp_pkts',
                'state': 'conn_state'   # Argus 'state' (CON, FIN) -> Zeek 'conn_state'
            }
            
            print("Consolidando colunas base (Argus > Zeek)...")
            for argus_col, zeek_col in consolidation_map.items():
                if argus_col in final_df.columns:
                    if zeek_col in final_df.columns:
                        # Usa combine_first: Se Argus tem valor, usa Argus. Se NaN, mantém Zeek.
                        # Mas como queremos FORÇAR Argus se disponível (pois Zeek não tem TTL), 
                        # podemos inverter ou usar fillna com cuidado. 
                        # Aqui usamos Argus para preencher buracos E sobrescrever se necessário? 
                        # Melhor estratégia para seu caso: Se Argus trouxe a coluna, confie no Argus para métricas de rede.
                        final_df[zeek_col] = final_df[argus_col].fillna(final_df[zeek_col])
                        
                        # Remove a coluna duplicada _argus ou original argus para limpar
                        final_df.drop(columns=[argus_col], inplace=True)
                    else:
                        # Se Zeek não tem, renomeia Argus para o nome esperado
                        final_df.rename(columns={argus_col: zeek_col}, inplace=True)
            
            # Garante preenchimento de zeros nas features exclusivas do Argus que não deram match
            new_features = ['sjit', 'djit', 'sinpkt', 'dinpkt', 'tcprtt', 'synack', 'ackdat', 'stcpb', 'dtcpb', 'swin', 'dwin', 'sttl', 'dttl', 'sloss', 'dloss']
            for feat in new_features:
                if feat in final_df.columns:
                    final_df[feat] = final_df[feat].fillna(0)

        else:
             print("AVISO: Nenhum arquivo Argus foi carregado com sucesso. Features complexas serão 0.")

        # 4. Rótulos e Exportação
        final_df['attack_cat'] = "Exploits"
        final_df['label'] = label_value
        
        # Limpeza de colunas temporárias
        cols_to_drop = [c for c in final_df.columns if c.endswith('_argus')]
        final_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

        print("\n--- Amostra do DataFrame Final (Verifique colunas sttl/dttl) ---")
        # Mostra colunas críticas para verificar se o merge funcionou
        cols_preview = ['ts', 'uid', 'id.orig_h', 'sttl', 'dttl', 'rate']
        cols_exist = [c for c in cols_preview if c in final_df.columns]
        display(final_df[cols_exist].head())

    else:
        print("ERRO: conn.log vazio ou não encontrado.")

Diretório base do script: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\exploits
--- Processando Categoria: Normal ---
Diretório Zeek: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\exploits\logs\logs
Diretório Argus: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\exploits\logs
Lidos 4 logs de conn. Total linhas: 278010
Lidos 2 logs de http. Total linhas: 277642
Lidos 3 logs de dns. Total linhas: 338

Iniciando carregamento dos arquivos Argus...
Carregando dados do Argus: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\exploits\logs\captura_exploit1.csv
Carregando dados do Argus: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\exploits\logs\captura_exploit2.csv

PADRONIZANDO CHAVES DE MERGE ZEEK <-> ARGUS...
DIAGNÓSTICO: Encontrados 221 fluxos coincidentes exatos entre Zeek e Argus.
Total de registros Arg

,ts,uid,id.orig_h,sttl,dttl
0,1.764260e+09,CKGF0h3Mj99toUjwpe,192.168.255.112,0.0,0.0
1,1.764260e+09,CuG9q92OapYB1UmNz4,192.168.255.112,0.0,0.0
2,1.764260e+09,CwnNox3nUBO3tJiY05,192.168.255.112,0.0,0.0
3,1.764260e+09,CgB2LdhdnVoCmY4ae,192.168.255.112,0.0,0.0
4,1.764260e+09,CYYUI84qCD26dEPkt1,192.168.255.112,0.0,0.0


In [2]:
# --- ENGENHARIA DE FEATURES (CÁLCULOS SIMPLES) ---
# Assume que 'final_df' existe da célula anterior

print("\n--- Iniciando Engenharia de Features Simples ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # 1. Tratar valores numéricos que podem ser string ou NaN antes dos cálculos
    numeric_cols_to_clean = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']
    for col in numeric_cols_to_clean:
        if col in final_df.columns:
            # Converte para numérico, erros viram NaN. Preenche NaN com 0.
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)
        else:
            print(f"Aviso: Coluna necessária '{col}' não encontrada para cálculos.")
            # Cria coluna com zeros se não existir para evitar erros posteriores
            final_df[col] = 0

    # 2. Calcular 'rate'
    # Evita divisão por zero: np.divide(..., where=denominator!=0)
    total_pkts = final_df['orig_pkts'] + final_df['resp_pkts']
    final_df['rate'] = np.divide(total_pkts, final_df['duration'], \
                                 out=np.zeros_like(total_pkts, dtype=float), where=final_df['duration']!=0)

    # 3. Calcular 'sload' (Source Load in bits per second)
    final_df['sload'] = np.divide(final_df['orig_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['duration']!=0)

    # 4. Calcular 'dload' (Destination Load in bits per second)
    final_df['dload'] = np.divide(final_df['resp_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['duration']!=0)

    # 5. Calcular 'smean' (Source Mean Packet Size)
    final_df['smean'] = np.divide(final_df['orig_bytes'], final_df['orig_pkts'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['orig_pkts']!=0).astype(int) # Usually integer

    # 6. Calcular 'dmean' (Destination Mean Packet Size)
    final_df['dmean'] = np.divide(final_df['resp_bytes'], final_df['resp_pkts'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['resp_pkts']!=0).astype(int) # Usually integer

    # 7. Calcular 'is_sm_ips_ports' (Source=Dest IP and Port)
    # Verifica se as colunas existem antes de comparar
    if 'id.orig_h' in final_df.columns and 'id.resp_h' in final_df.columns and \
       'id.orig_p' in final_df.columns and 'id.resp_p' in final_df.columns:
        final_df['is_sm_ips_ports'] = ((final_df['id.orig_h'] == final_df['id.resp_h']) & \
                                       (final_df['id.orig_p'] == final_df['id.resp_p'])).astype(int)
    else:
        print("Aviso: Colunas de IP/Porta não encontradas. 'is_sm_ips_ports' será 0.")
        final_df['is_sm_ips_ports'] = 0

    # 8. Calcular 'is_ftp_login'
    # Verifica se as colunas do FTP (resultantes do merge) existem
    if 'ftp_user' in final_df.columns and 'ftp_password' in final_df.columns:
        # Será 1 se AMBOS user e password não forem NaN (ou seja, foram preenchidos no log)
        final_df['is_ftp_login'] = ((final_df['ftp_user'].notna()) & \
                                    (final_df['ftp_password'].notna())).astype(int)
    else:
        # Se não houve merge com ftp.log ou as colunas não existiam
        print("Aviso: Colunas 'ftp_user'/'ftp_password' não encontradas. 'is_ftp_login' será 0.")
        final_df['is_ftp_login'] = 0


    print("\n--- Amostra do DataFrame Após Adicionar Features Simples ---")
    display(final_df[['uid', 'duration', 'orig_pkts', 'resp_pkts', 'orig_bytes', 'resp_bytes', \
                      'rate', 'sload', 'dload', 'smean', 'dmean', 'is_sm_ips_ports', 'is_ftp_login', \
                      'attack_cat', 'label']].head()) # Mostra apenas algumas colunas chave + as novas
    

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute a célula anterior primeiro.")


--- Iniciando Engenharia de Features Simples ---
Aviso: Colunas 'ftp_user'/'ftp_password' não encontradas. 'is_ftp_login' será 0.

--- Amostra do DataFrame Após Adicionar Features Simples ---


,uid,duration,orig_pkts,resp_pkts,orig_bytes,resp_bytes,rate,sload,dload,smean,dmean,is_sm_ips_ports,is_ftp_login,attack_cat,label
0,CKGF0h3Mj99toUjwpe,0.015160,5.0,0.0,410.0,0.0,329.815303,216358.839050,0.0,82,0,0,0,Exploits,1
1,CuG9q92OapYB1UmNz4,0.034920,5.0,0.0,379.0,0.0,143.184422,86827.033219,0.0,75,0,0,0,Exploits,1
2,CwnNox3nUBO3tJiY05,0.021723,5.0,0.0,674.0,0.0,230.170787,248216.176403,0.0,134,0,0,0,Exploits,1
3,CgB2LdhdnVoCmY4ae,0.004938,5.0,0.0,379.0,0.0,1012.555691,614013.770757,0.0,75,0,0,0,Exploits,1
4,CYYUI84qCD26dEPkt1,0.432101,5.0,0.0,410.0,0.0,11.571369,7590.817887,0.0,82,0,0,0,Exploits,1


In [3]:
# --- ENGENHARIA DE FEATURES (AGREGAÇÕES ct_*) ---
# Assume que 'final_df' existe das células anteriores

print("\n--- Iniciando Engenharia de Features Agregadas (ct_*) ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # PASSO 1: Garantir que os dados estejam ordenados por Timestamp
    print("Ordenando DataFrame por timestamp...")
    df_sorted = final_df.sort_values(by='ts').reset_index(drop=True)

    # PASSO 2: Definir o tamanho da janela
    window_size = 100
    print(f"Usando uma janela deslizante de {window_size} conexões.")

    # PASSO 3: Calcular as features ct_*

    # --- Features baseadas em IP/Porta/Serviço (Loop iterrows) ---
    results = {}
    print("Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...")
    if 'id.resp_h' in df_sorted.columns and 'id.resp_p' in df_sorted.columns:
        df_sorted['dst_ip_port'] = df_sorted['id.resp_h'].astype(str) + ':' + df_sorted['id.resp_p'].astype(str)
    if 'id.orig_h' in df_sorted.columns and 'id.orig_p' in df_sorted.columns:
        df_sorted['src_ip_port'] = df_sorted['id.orig_h'].astype(str) + ':' + df_sorted['id.orig_p'].astype(str)

    num_rows = len(df_sorted)
    
    for i, row in df_sorted.iterrows():
        start_idx = max(0, i - window_size + 1)
        current_window = df_sorted.iloc[start_idx : i + 1]

        # Calcula cada feature (lógica idêntica à anterior)
        if 'id.orig_h' in row and 'id.resp_p' in row and 'dst_ip_port' in row:
             results.setdefault('ct_srv_src', []).append(current_window[ (current_window['dst_ip_port'] == row['dst_ip_port']) & \
                                                                         (current_window['id.orig_h'] == row['id.orig_h']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row and 'src_ip_port' in row:
             results.setdefault('ct_srv_dst', []).append(current_window[ (current_window['src_ip_port'] == row['src_ip_port']) & \
                                                                         (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])
        if 'id.resp_h' in row:
             results.setdefault('ct_dst_ltm', []).append(current_window[ current_window['id.resp_h'] == row['id.resp_h'] ].shape[0])
        if 'id.orig_h' in row:
             results.setdefault('ct_src_ltm', []).append(current_window[ current_window['id.orig_h'] == row['id.orig_h'] ].shape[0])
        if 'id.orig_h' in row and 'id.resp_p' in row:
             results.setdefault('ct_src_dport_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                               (current_window['id.resp_p'] == row['id.resp_p']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row:
             results.setdefault('ct_dst_sport_ltm', []).append(current_window[ (current_window['id.resp_h'] == row['id.resp_h']) & \
                                                                               (current_window['id.orig_p'] == row['id.orig_p']) ].shape[0])
        if 'id.orig_h' in row and 'id.resp_h' in row:
             results.setdefault('ct_dst_src_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                             (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])

        if (i + 1) % 500 == 0:
            print(f"  Processado {i + 1}/{num_rows} linhas...")

    print("  Cálculo das features baseadas em IP/Porta/Serviço concluído.")
    for feature_name, values in results.items():
         if len(values) == len(df_sorted):
              df_sorted[feature_name] = values
         else:
              print(f"Erro de tamanho: '{feature_name}'. Preenchendo com 0.")
              df_sorted[feature_name] = 0
    df_sorted = df_sorted.drop(columns=['dst_ip_port', 'src_ip_port'], errors='ignore')

    # -- Features Específicas (ct_state_ttl, ct_ftp_cmd, ct_flw_http_mthd) --

    # ct_state_ttl (Contagem por estado) - Usando Factorization e indexação NumPy
    print("Calculando ct_state_ttl (usando factorization)...")
    if 'sttl' in df_sorted.columns and 'dttl' in df_sorted.columns and 'conn_state' in df_sorted.columns:
        # SE tivermos dados do Argus (sttl não é tudo zero ou nulo)
        if df_sorted['sttl'].sum() > 0: 
            print("  Usando lógica REAL do UNSW-NB15 (State + STTL + DTTL)...")
            # Agrupa por State e TTLs
            # OBS: Convertendo para str para agrupar, pois NaN ou float atrapalham groupby
            df_sorted['sttl'] = df_sorted['sttl'].fillna(0).astype(int)
            df_sorted['dttl'] = df_sorted['dttl'].fillna(0).astype(int)
            
            # A lógica original do UNSW é contar a frequência dessa combinação
            # Usamos transform('count') para aplicar a contagem em cada linha
            df_sorted['ct_state_ttl'] = df_sorted.groupby(['conn_state', 'sttl', 'dttl'])['conn_state'].transform('count')
        else:
            print("  Colunas STTL existem mas estão vazias/zeradas. Usando Fallback (Apenas State)...")
            # Lógica de fallback original (apenas conn_state)
            state_codes, state_uniques = pd.factorize(df_sorted['conn_state'])
            df_sorted['_state_codes'] = state_codes
            ct_state_ttl_result = df_sorted.groupby('conn_state')['_state_codes'] \
                                            .rolling(window=window_size, min_periods=1) \
                                            .apply(lambda x: (x == x[-1]).sum(), raw=True) 
            ct_state_ttl_result = ct_state_ttl_result.reset_index(level=0, drop=True).sort_index().fillna(1).astype(int)
            df_sorted['ct_state_ttl'] = ct_state_ttl_result
            df_sorted = df_sorted.drop(columns=['_state_codes'])
            
    elif 'conn_state' in df_sorted.columns:
        print("  Dados de TTL ausentes. Usando Fallback (Apenas State)...")
        # Copia da lógica original de fallback
        state_codes, state_uniques = pd.factorize(df_sorted['conn_state'])
        df_sorted['_state_codes'] = state_codes
        ct_state_ttl_result = df_sorted.groupby('conn_state')['_state_codes'] \
                                        .rolling(window=window_size, min_periods=1) \
                                        .apply(lambda x: (x == x[-1]).sum(), raw=True)
        ct_state_ttl_result = ct_state_ttl_result.reset_index(level=0, drop=True).sort_index().fillna(1).astype(int)
        df_sorted['ct_state_ttl'] = ct_state_ttl_result
        df_sorted = df_sorted.drop(columns=['_state_codes'])
    else:
        print("Aviso: Coluna 'conn_state' não encontrada. 'ct_state_ttl' será 0.")
        df_sorted['ct_state_ttl'] = 0

    # ct_ftp_cmd (Lógica idêntica à anterior)
    print("Calculando ct_ftp_cmd...")
    if 'ftp_command' in df_sorted.columns:
        df_sorted['ct_ftp_cmd'] = df_sorted['ftp_command'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'ftp_command' não encontrada. 'ct_ftp_cmd' será 0.")
        df_sorted['ct_ftp_cmd'] = 0

    # ct_flw_http_mthd (Lógica idêntica à anterior)
    print("Calculando ct_flw_http_mthd...")
    if 'http_method' in df_sorted.columns:
        df_sorted['ct_flw_http_mthd'] = df_sorted['http_method'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'http_method' não encontrada. 'ct_flw_http_mthd' será 0.")
        df_sorted['ct_flw_http_mthd'] = 0

    print("\n--- Amostra do DataFrame Após Adicionar Features Agregadas ---")
    ct_cols_to_show = [col for col in df_sorted.columns if col.startswith('ct_')]
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].head())
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].tail())

    final_engineered_df = df_sorted # Atribui o resultado final
    # --- Exportar para CSV (Opcional) ---
    output_filename = f"{"exploits".lower().replace(os.path.sep, '_')}_processed.csv" # Nomeia arquivo baseado na categoria
    final_engineered_df.to_csv(os.path.join(base_project_path, output_filename), index=False)
    print(f"\nDataFrame exportado para {output_filename}")

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute as células anteriores primeiro.")


--- Iniciando Engenharia de Features Agregadas (ct_*) ---
Ordenando DataFrame por timestamp...
Usando uma janela deslizante de 100 conexões.
Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...
  Processado 500/278010 linhas...
  Processado 1000/278010 linhas...
  Processado 1500/278010 linhas...
  Processado 2000/278010 linhas...
  Processado 2500/278010 linhas...
  Processado 3000/278010 linhas...
  Processado 3500/278010 linhas...
  Processado 4000/278010 linhas...
  Processado 4500/278010 linhas...
  Processado 5000/278010 linhas...
  Processado 5500/278010 linhas...
  Processado 6000/278010 linhas...
  Processado 6500/278010 linhas...
  Processado 7000/278010 linhas...
  Processado 7500/278010 linhas...
  Processado 8000/278010 linhas...
  Processado 8500/278010 linhas...
  Processado 9000/278010 linhas...
  Processado 9500/278010 linhas...
  Processado 10000/278010 linhas...
  Processado 10500/278010 linhas...
  Processado 11000/278010 linhas...
  Pr

,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
0,1.764260e+09,CN9fSKou3sx8GZkEg,192.168.255.112,192.168.255.221,1,1,1,1,1,1,1,13,0,0
1,1.764260e+09,CKGF0h3Mj99toUjwpe,192.168.255.112,192.168.255.221,1,1,2,2,1,1,2,277644,0,1
2,1.764260e+09,CuG9q92OapYB1UmNz4,192.168.255.112,192.168.255.221,2,1,3,3,2,1,3,277644,0,2
3,1.764260e+09,CwnNox3nUBO3tJiY05,192.168.255.112,192.168.255.221,3,1,4,4,3,1,4,277644,0,3
4,1.764260e+09,CgB2LdhdnVoCmY4ae,192.168.255.112,192.168.255.221,4,1,5,5,4,1,5,277644,0,4


,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
278005,1.764622e+09,CEcCuJ2hNRlo7Fixta,192.168.255.211,224.0.0.251,2,2,13,2,2,13,2,101,0,31
278006,1.764622e+09,CCawqQONSl9n4lEJe,192.168.255.158,224.0.0.251,4,4,14,4,4,14,4,101,0,31
278007,1.764622e+09,ClmpHM3PvDYlWGqVW8,192.168.255.112,192.168.255.221,32,1,32,33,32,1,32,277644,0,32
278008,1.764622e+09,CwsvbC49SHcljxRJ38,192.168.255.112,192.168.255.221,33,1,33,33,33,1,33,277644,0,33
278009,1.764622e+09,CBMsaK21sLp2Fy7DFb,192.168.255.112,192.168.255.221,1,1,34,34,1,1,34,13,0,33



DataFrame exportado para exploits_processed.csv
